# บทที่ 2 — บีบ 20,480 จุด ให้เหลือ 14 ตัวเลข

<sub>บทเรียนที่ 2 จาก 8 &nbsp;·&nbsp; [← บทที่ 1](01_signal_basics.ipynb) · [สารบัญ](README.md) · [บทที่ 3 →](03_building_labels.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เข้าใจว่าทำไมต้องสกัด feature แทนที่จะป้อนสัญญาณดิบ
- เขียนโค้ดคำนวณ RMS, Kurtosis, Crest Factor ด้วยตัวเอง
- รู้ว่าแต่ละ feature จับลักษณะอะไรของสัญญาณ
- เห็นว่า feature ตัวไหนตอบสนองต่อการเสื่อมเร็วที่สุด

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 2.1 ทำไมต้องสกัด feature

จากบทที่แล้ว หนึ่งไฟล์มี 20,480 จุดต่อลูกปืน ถ้าเอาทั้ง 984 ไฟล์:

```
984 ไฟล์ × 20,480 จุด × 4 ลูกปืน ≈ 80,000,000 ตัวเลข
```

ป้อนเข้าโมเดลลำดับเวลาตรง ๆ ไม่ไหวแน่นอน และส่วนใหญ่ก็เป็นข้อมูลซ้ำที่ไม่ให้ข้อมูลเพิ่ม

**ทางแก้:** บีบสัญญาณ 1 วินาทีให้เหลือตัวเลขไม่กี่ตัวที่สรุปลักษณะสำคัญไว้
เรียกว่า **Feature Extraction**

เราเลือก 14 ตัวที่วิศวกรเครื่องกลรู้อยู่แล้วว่าไวต่อความเสียหายของลูกปืน

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

## 2.2 เขียนเองก่อน แล้วค่อยดูของจริง

เริ่มจาก 3 ตัวที่สำคัญที่สุด มาเขียนสูตรเองเพื่อให้เข้าใจว่ามันวัดอะไร

In [ ]:
def rms(x):
    """Root Mean Square - 'พลังงานเฉลี่ย' ของสัญญาณ

    ยกกำลังสองก่อน (ทำให้ค่าลบเป็นบวก) เฉลี่ย แล้วถอดรากกลับ
    """
    return np.sqrt(np.mean(x ** 2))


def kurtosis(x):
    """Kurtosis - 'ความแหลม' ของการกระจาย

    ยกกำลัง 4 ทำให้ค่าที่อยู่ไกลค่าเฉลี่ยถูกขยายอย่างมหาศาล
    สัญญาณสุ่มปกติ (Gaussian) จะได้ค่าราว 3
    ถ้าสูงกว่านั้นมาก = มียอดโดดผิดปกติ
    """
    mu = np.mean(x)
    sigma = np.std(x)
    return np.mean((x - mu) ** 4) / (sigma ** 4)


def crest_factor(x):
    """Crest Factor - ยอดสูงกว่าค่าเฉลี่ยกี่เท่า

    ค่าสูง = สัญญาณมียอดโดดขึ้นมาจากพื้น = มีการกระแทก
    """
    return np.max(np.abs(x)) / rms(x)


# ทดสอบกับสัญญาณที่เรารู้คำตอบอยู่แล้ว
rng = np.random.default_rng(0)
noise = rng.normal(0, 1, 20000)               # สัญญาณรบกวนธรรมดา
spiky = noise.copy()
spiky[::500] += rng.normal(0, 8, len(spiky[::500]))   # แอบใส่ยอดแหลมเข้าไป

print(f"{'':22} {'RMS':>8} {'Kurtosis':>10} {'Crest':>8}")
print("-" * 52)
print(f"{'สัญญาณรบกวนธรรมดา':<22} {rms(noise):>8.3f} {kurtosis(noise):>10.3f} {crest_factor(noise):>8.3f}")
print(f"{'มียอดแหลมแทรก':<22} {rms(spiky):>8.3f} {kurtosis(spiky):>10.3f} {crest_factor(spiky):>8.3f}")

**นี่คือบทเรียนสำคัญที่สุดของบทนี้** — ดูตัวเลขที่ได้:

- **RMS** เพิ่มขึ้นนิดเดียว เพราะยอดแหลมมีไม่กี่จุดเมื่อเทียบกับ 20,000 จุด
  พอเฉลี่ยแล้วแทบไม่ขยับ
- **Kurtosis** พุ่งขึ้นหลายเท่า เพราะการยกกำลัง 4 ขยายค่าที่อยู่ไกลค่าเฉลี่ยอย่างรุนแรง
- **Crest Factor** ก็เพิ่มชัดเจน

แปลว่า **Kurtosis กับ Crest Factor จับ "รอยแตกเล็ก ๆ" ได้ตั้งแต่ RMS ยังแทบไม่ขยับ**

ในทางปฏิบัติหมายความว่าเราจะได้สัญญาณเตือนเร็วขึ้นมาก

## 2.3 feature ครบทั้ง 14 ตัว

โปรเจกต์นี้ใช้ 14 ตัวต่อลูกปืน แบ่งเป็นสองกลุ่ม

**กลุ่มโดเมนเวลา (10 ตัว)** — คำนวณจากสัญญาณตรง ๆ

| # | Feature | สูตร | จับอะไร |
|---|---|---|---|
| 1 | RMS | √(Σx²/N) | พลังงานรวม |
| 2 | Peak | max(\|x\|) | ค่าสูงสุด |
| 3 | Peak-to-Peak | max − min | ช่วงกว้าง |
| 4 | Crest Factor | Peak / RMS | ความโดดของยอด |
| 5 | Kurtosis | E[(x−μ)⁴]/σ⁴ | ความแหลม |
| 6 | Skewness | E[(x−μ)³]/σ³ | ความเบ้ |
| 7 | Shape Factor | RMS / mean\|x\| | รูปคลื่น |
| 8 | Impulse Factor | Peak / mean\|x\| | ความรุนแรงการกระแทก |
| 9 | Margin Factor | Peak / (mean√\|x\|)² | ไวต่อการสึกช่วงต้น |
| 10 | Std | σ(x) | ความผันแปร |

**กลุ่มโดเมนความถี่ (4 ตัว)** — ต้องทำ FFT ก่อน

| # | Feature | ย่าน | จับอะไร |
|---|---|---|---|
| 11-13 | Band Energy | ต่ำ / กลาง / สูง | พลังงานแยกตามย่าน |
| 14 | Spectral Centroid | ทั้งหมด | จุดศูนย์ถ่วงความถี่ |

**รวม: 14 features × 4 ลูกปืน = 56 features ต่อจุดเวลา**

ข้อมูลถูกบีบจาก ~80 ล้านตัวเลข เหลือตาราง `(984, 56)` = 55,104 ตัวเลข — **เล็กลงกว่า 1,400 เท่า**

## 2.4 ใช้ของจริงจาก `src/features.py`

โค้ดจริงของโปรเจกต์อยู่ที่ `src/features.py` เราไม่ต้องคำนวณใหม่ทั้งหมด
เพราะผลลัพธ์ถูก cache ไว้แล้วที่ `outputs/features_cache.npz`

In [ ]:
from src.paths import FEATURES_CACHE

features = np.load(FEATURES_CACHE)["features"]
print("รูปร่างตาราง features:", features.shape, " (984 จุดเวลา x 56 features)")

# ลำดับ feature ใน 1 ลูกปืน (ดูได้จาก src/features.py)
FEATURE_NAMES = ["RMS", "Peak", "P2P", "Crest", "Kurtosis", "Skewness",
                 "Shape", "Impulse", "Margin", "Std",
                 "BandLow", "BandMid", "BandHigh", "Centroid"]

# feature ของ Bearing k อยู่ที่ index k*14 ถึง k*14+13
def idx(bearing, feature_name):
    """หา column index ของ feature ที่ต้องการ (bearing เริ่มที่ 0)"""
    return bearing * 14 + FEATURE_NAMES.index(feature_name)


print("\nตัวอย่าง: RMS ของ Bearing 3 อยู่ที่คอลัมน์", idx(2, "RMS"))
print("         Kurtosis ของ Bearing 3 อยู่ที่คอลัมน์", idx(2, "Kurtosis"))

## 2.5 feature ตัวไหนเตือนเราก่อน

นี่คือคำถามที่สำคัญที่สุดในทางปฏิบัติ — มาพล็อตเทียบกัน

เพื่อให้เทียบกันได้ทั้งที่หน่วยต่างกัน เราจะ normalize ทุกเส้นให้อยู่ในช่วง 0–1

In [ ]:
def normalize(v):
    """ปรับค่าให้อยู่ในช่วง 0-1 เพื่อให้เทียบรูปร่างของเส้นได้"""
    return (v - v.min()) / (v.max() - v.min() + 1e-12)


BEARING = 0   # Bearing 1 - ตัวที่พัง
watch = ["RMS", "Kurtosis", "Crest", "BandHigh"]
colors = ["#1976d2", "#d32f2f", "#f57c00", "#7b1fa2"]

plt.figure(figsize=(11, 4.5))
for name, c in zip(watch, colors):
    plt.plot(normalize(features[:, idx(BEARING, name)]),
             label=name, color=c, linewidth=1.2, alpha=0.85)

plt.xlabel("จุดเวลา (ห่างกัน 10 นาที)")
plt.ylabel("ค่าที่ normalize แล้ว (0-1)")
plt.title("Bearing 1 - feature ตัวไหนตอบสนองต่อการเสื่อมก่อน")
plt.legend()
plt.tight_layout()
plt.show()

ลองสังเกตว่าเส้นไหนเริ่มขยับก่อน แล้วมาวัดกันด้วยตัวเลข

In [ ]:
# นิยาม "เริ่มเตือน" = ค่าเกิน 30% ของช่วงทั้งหมดเป็นครั้งแรก
THRESHOLD = 0.30

print(f"{'Feature':<12} {'เตือนที่จุดเวลา':>16} {'= ชั่วโมงที่':>14} {'ก่อนจบการทดลอง':>18}")
print("-" * 66)

total_steps = len(features)
rows = []
for name in FEATURE_NAMES:
    v = normalize(features[:, idx(BEARING, name)])
    over = np.where(v > THRESHOLD)[0]
    if len(over):
        t = int(over[0])
        rows.append((t, name))

for t, name in sorted(rows):
    hours = t * 10 / 60
    before = (total_steps - t) * 10 / 60
    print(f"{name:<12} {t:>16} {hours:>13.1f}h {before:>17.1f}h")

ยิ่งตัวเลขคอลัมน์สุดท้ายมาก = ยิ่งเตือนล่วงหน้านาน = ยิ่งมีประโยชน์

**ข้อสรุปที่ควรจำ:** ไม่มี feature ตัวไหนดีที่สุดตัวเดียว
บางตัวเตือนเร็วแต่มีสัญญาณรบกวนเยอะ บางตัวนิ่งแต่เตือนช้า
**เราจึงป้อนทั้ง 56 ตัวให้โมเดลไปเรียนรู้เองว่าจะชั่งน้ำหนักอย่างไร**

## 2.6 ลูกปืนที่ปกติเป็นอย่างไร

เพื่อให้เห็นภาพชัด ลองเทียบ RMS ของทั้ง 4 ลูกปืน

In [ ]:
labels = ["Bearing 1 (วงแหวนนอกแตก)", "Bearing 2 (ปกติ)",
          "Bearing 3 (ปกติ)", "Bearing 4 (ปกติ)"]
colors = ["#d32f2f", "#388e3c", "#1976d2", "#f57c00"]

plt.figure(figsize=(11, 4.5))
for b in range(4):
    plt.plot(features[:, idx(b, "RMS")], label=labels[b], color=colors[b], linewidth=1.1)

plt.xlabel("จุดเวลา (ห่างกัน 10 นาที)")
plt.ylabel("RMS")
plt.title("RMS ของทั้ง 4 ลูกปืนตลอดการทดลอง")
plt.legend()
plt.tight_layout()
plt.show()

กราฟนี้คือภาพรวมของทั้งการทดลอง — สามเส้นค่อนข้างนิ่งตลอด
ส่วนเส้นสีแดง (Bearing 1) พุ่งขึ้นชัดเจนในช่วงท้าย

และนี่คือ**รูปแบบการเสื่อมที่แท้จริง**: นิ่งเกือบตลอดแล้วทรุดเร็วช่วงท้าย
ไม่ใช่ลดลงเป็นเส้นตรง — จำจุดนี้ไว้ให้ดี เพราะบทที่ 3 จะใช้

## 🔧 ลองแก้ดู — สำรวจ feature ตัวอื่น


1. เปลี่ยน `BEARING = 0` เป็น `BEARING = 1` (ลูกปืนที่ปกติดี) แล้วรันเซลล์กราฟใหม่
   — เส้นควรจะนิ่งกว่ามาก ถ้ายังกระโดดแปลว่าอะไร?
2. เพิ่ม `"Impulse"` และ `"Margin"` เข้าไปในลิสต์ `watch` แล้วดูว่าเตือนเร็วแค่ไหน
3. ลองเปลี่ยน `THRESHOLD` เป็น `0.15` และ `0.50` — อันดับของ feature เปลี่ยนไหม?
   ถ้าเปลี่ยน แปลว่าการเลือก threshold มีผลต่อข้อสรุปแค่ไหน?

## ❓ เช็คความเข้าใจ

**1. ทำไม Kurtosis ถึงจับรอยแตกเล็ก ๆ ได้ก่อน RMS?**

<details>
<summary>ดูเฉลย</summary>

เพราะ Kurtosis ยกกำลัง 4 ซึ่งขยายค่าที่อยู่ไกลจากค่าเฉลี่ยอย่างรุนแรง ยอดแหลมไม่กี่จุดจึงมีผลต่อค่ามาก ในขณะที่ RMS ยกกำลังแค่ 2 แล้วเฉลี่ยกับข้อมูล 20,000 จุด ยอดแหลมไม่กี่จุดจึงถูกกลืนหายไป

</details>

**2. ถ้าเราป้อนสัญญาณดิบ 20,480 จุดเข้า LSTM ตรง ๆ จะเกิดอะไรขึ้น?**

<details>
<summary>ดูเฉลย</summary>

หน่วยความจำจะไม่พอ เทรนช้ามาก และ LSTM จะจำ dependency ข้ามหลักหมื่น timestep ไม่ไหว (gradient จะหายไประหว่างทาง) ที่สำคัญคือข้อมูลส่วนใหญ่เป็นสัญญาณรบกวนซ้ำ ๆ การสกัด feature ที่มีความหมายทางฟิสิกส์ให้ผลดีกว่ามาก

</details>

**3. ทำไมต้อง normalize ก่อนพล็อตเทียบ feature หลายตัว?**

<details>
<summary>ดูเฉลย</summary>

เพราะแต่ละ feature มีหน่วยและช่วงค่าต่างกันมาก เช่น RMS อาจอยู่ราว 0.1 แต่ Kurtosis อาจเป็นหลักสิบ ถ้าพล็อตรวมกันโดยไม่ปรับสเกล เส้นที่ค่าน้อยจะแบนติดแกน จนดูรูปร่างไม่ออก

</details>

---

## สรุปบทนี้

- สกัด 14 features ต่อลูกปืน รวม 56 features ต่อจุดเวลา — เล็กลงกว่า 1,400 เท่า
- Kurtosis และ Crest Factor ตอบสนองต่อรอยแตกเร็วกว่า RMS
- ไม่มี feature ตัวไหนดีที่สุดตัวเดียว จึงป้อนทั้งหมดให้โมเดลชั่งน้ำหนักเอง
- การเสื่อมจริงคือ 'นิ่งนาน แล้วทรุดเร็ว' ไม่ใช่เส้นตรง

[← บทที่ 1](01_signal_basics.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 3 — สร้าง Label →](03_building_labels.ipynb)**